# SFT-пайплайн: BabyAI (Minari) → RGB replay → nanoVLM fine-tuning

Полный цикл (Variant A):
1. **Minari** — скачать BabyAI maze/navigation expert trajectories (`minari download`)
2. **RGB replay** — воспроизвести эпизоды с `RGBImgPartialObsWrapper` (см. `minari_adapter.py`)
3. **Fine-tuning** предобученного `nanoVLM-222M` (7 действий BabyAI)
4. **Метрики** в консоль + графики loss / accuracy (episode-level val split)
5. **Сохранение** чекпоинта и пробный inference

> Запускайте ячейки сверху вниз. Kernel: `.venv` проекта. Нужны: `minari`, `minigrid`, `h5py`.

In [ ]:
!git clone https://github.com/berkutivan/NanoVLM
%cd /content/NanoVLM
!unzip /content/drive/MyDrive/Tbank/checkpoints.zip
!pip install -r requirements.txt -quiet
%cd /content/NanoVLM/Training_model

## 0. Пути и зависимости

In [ ]:
import json
import math
import os
import random
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader

os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Корень репозитория Tbank
ROOT = Path.cwd().resolve()
if ROOT.name == "Training_model":
    ROOT = ROOT.parent

DATASETS_DIR = ROOT / "Datasets"
NANOVLM_DIR = ROOT / "nanoVLM"
TRAINING_DIR = ROOT / "Training_model"
DATASET_JSON = DATASETS_DIR / "dataset.json"
PRETRAINED_CKPT = ROOT / "checkpoints" / "nanoVLM-222M"
SFT_OUTPUT_DIR = ROOT / "checkpoints" / "sft-minigrid"

for p in (str(NANOVLM_DIR), str(DATASETS_DIR), str(TRAINING_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)

print("ROOT:", ROOT)
print("CUDA:", torch.cuda.is_available(), "| device:", "cuda" if torch.cuda.is_available() else "cpu")

## 1. Генерация `dataset.json`

Каждый объект = одно процедурное поле (`ProceduralMazeEnv`):
- `layout` — стены, цель, старт агента
- `mission` — текст задачи
- `predicate_space` — действия, размер сетки, формат RGB-наблюдения

Curriculum: сначала маленькие карты без стен, затем сложнее.

In [ ]:
from maze_presample import (
    CURRICULUM,
    MazePreSampleBuilder,
    MazePreSampleConfig,
    load_dataset,
    save_dataset,
    curriculum_for_index,
)


def generate_dataset(n_mazes: int, overwrite: bool = False) -> dict:
    """Создать или дополнить dataset.json с n_mazes лабиринтами."""
    if overwrite or not DATASET_JSON.exists():
        data = {"version": 2, "format": "pre_sample", "objects": []}
    else:
        data = load_dataset()

    start_id = len(data["objects"])
    target = start_id + n_mazes

    while len(data["objects"]) < target:
        obj_id = len(data["objects"])
        cur = curriculum_for_index(obj_id)
        cfg = MazePreSampleConfig(
            size=cur["size"],
            n_walls=cur["n_walls"],
            seed=obj_id,
        )
        sample = MazePreSampleBuilder(cfg).build()
        sample["id"] = obj_id
        data["objects"].append(sample)
        print(
            f"  id={obj_id} size={cur['size']} walls={cur['n_walls']} "
            f"goal={sample['layout']['goal_pos']}"
        )

    save_dataset(data)
    print(f"Saved {len(data['objects'])} objects -> {DATASET_JSON}")
    return data


# --- настройка ---
N_MAZES = 100          # сколько лабиринтов (100 уже есть? поставьте 0 и skip)
OVERWRITE = False      # True = пересоздать json с нуля

if N_MAZES > 0 or OVERWRITE:
    print("Generating dataset...")
    dataset = generate_dataset(N_MAZES, overwrite=OVERWRITE)
else:
    dataset = load_dataset()
    print(f"Using existing dataset: {len(dataset['objects'])} objects")

In [ ]:
# Быстрый просмотр одного объекта
obj = dataset["objects"][0]
print(json.dumps({k: obj[k] for k in ("id", "mission", "layout", "predicate_space")}, indent=2))

## 2. Визуализация поля и экспертная траектория (BFS)

Эксперт: **BFS** по состоянию `(x, y, direction)` — кратчайший путь в лабиринте.
Кадры RGB снимаются через `RGBImgPartialObsWrapper` **на лету** (не хранятся в json).

In [ ]:
from maze_expert import plan_expert_actions, rollout_expert_trajectory

demo_obj = dataset["objects"][0]
actions = plan_expert_actions(demo_obj)
trajectory = rollout_expert_trajectory(demo_obj)

print(f"Expert path length: {len(actions)} steps")
print("Actions:", " -> ".join(actions))

fig, axes = plt.subplots(1, min(5, len(trajectory)), figsize=(3 * min(5, len(trajectory)), 3))
if len(trajectory) == 1:
    axes = [axes]
for ax, step in zip(axes, trajectory[:5]):
    ax.imshow(step.image)
    ax.set_title(f"step {step.step_index}: {step.action}")
    ax.axis("off")
plt.suptitle(demo_obj["mission"])
plt.tight_layout()
plt.show()

## 3. Загрузка предобученного nanoVLM-222M

Веса должны лежать в `checkpoints/nanoVLM-222M/` (`config.json` + `model.safetensors`).
Если папки нет — скачаем с Hugging Face Hub.

In [ ]:
from models.vision_language_model import VisionLanguageModel
from data.processors import get_image_processor, get_tokenizer

HUB_ID = "lusxvr/nanoVLM-222M"
source = str(PRETRAINED_CKPT) if PRETRAINED_CKPT.exists() else HUB_ID

print(f"Loading VLM from: {source}")
model = VisionLanguageModel.from_pretrained(source)

# Colab workaround (PyTorch SDPA):
# Some torch versions error when passing an explicit attn_mask together with is_causal=True.
# We keep padding masks enabled by forcing nanoVLM LM attention to use the manual path.
if hasattr(model, "decoder") and hasattr(model.decoder, "blocks"):
    n = 0
    for blk in model.decoder.blocks:
        if hasattr(blk, "attn") and hasattr(blk.attn, "sdpa"):
            blk.attn.sdpa = False
            n += 1
    print(f"Disabled SDPA in {n} LM blocks")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")

tokenizer = get_tokenizer(model.cfg.lm_tokenizer)
image_processor = get_image_processor(model.cfg.vit_img_size)
print("Tokenizer:", model.cfg.lm_tokenizer)
print("Max text length:", model.cfg.lm_max_length)

In [ ]:
# Тестовое предсказание (pretrained VLM) на одном шаге эксперта
import torch
from PIL import Image

from maze_expert import rollout_expert_trajectory

if "dataset" not in globals():
    dataset = load_dataset()

demo_obj = dataset["objects"][0]
step0 = rollout_expert_trajectory(demo_obj)[0]

prompt = (
    f"Mission: {step0.mission}\n"
    "What is the next action? Answer with one word: left, right, or forward.\n"
    "Answer:"
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

enc = tokenizer([prompt], padding=True, padding_side="left", return_tensors="pt",
                truncation=True, max_length=model.cfg.lm_max_length)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

img = Image.fromarray(step0.image).convert("RGB")
img_t = image_processor(img).unsqueeze(0).to(device)

from action_logloss import action_first_token_ids

first_action_toks = action_first_token_ids(tokenizer, device)
with torch.no_grad():
    gen = model.generate(
        input_ids,
        img_t,
        attention_mask,
        max_new_tokens=8,
        do_sample=False,
        restrict_first_token_to=first_action_toks,
    )

pred = tokenizer.batch_decode(gen, skip_special_tokens=True)[0]


from action_logloss import ACTION_ORDER, action_first_token_ids, actions_from_first_token_ids

def first_grid_action(text: str) -> str:
    t = (text or "").strip().lower().split()
    if not t:
        return ""
    w = t[0].rstrip(".,;:!?")
    return w if w in ACTION_ORDER else ""

print("Mission:", step0.mission)
print("Gold action:", step0.action)
print("Model raw output:", pred)
print("First grid action token:", first_grid_action(pred))

## 4. SFT-датасет (Minari) и collator

Каждый **шаг эпизода BabyAI** = один training sample:
- **Источник:** Minari HDF5 (`minigrid/BabyAI-*/optimal-v0`), expert bot
- **RGB:** replay action sequence + `RGBImgPartialObsWrapper` (не compact 7×7 из HDF5)
- **Вход:** RGB-кадр + промпт с mission
- **Цель:** одно слово из `{left, right, forward, pickup, drop, toggle, done}`
- **Split:** по `episode_id` (без утечки шагов между train/val)
- **Loss:** embedding log-loss + CE на answer tokens (`action_logloss.py`)

In [ ]:
from sft_config import SFTConfig, DEFAULT_MINARI_DATASETS
from minigrid_sft_dataset import (
    MiniGridSFTDataset,
    filter_minari_cache,
    precompute_minari_for_keys,
    split_minari_episodes,
    PROMPT_TEMPLATE,
)
from minari_adapter import iterate_episode_ids, load_minari_dataset, MINARI_MAZE_DATASETS
import importlib
import minigrid_collator as _mc
importlib.reload(_mc)
from minigrid_collator import MiniGridSFTCollator

cfg = SFTConfig()
MINARI_DATASETS = list(DEFAULT_MINARI_DATASETS)  # или MINARI_MAZE_DATASETS для всех maze env
MINARI_DOWNLOAD = True
MAX_EPISODES_PER_DATASET = None  # smoke: 5
VAL_RATIO = cfg.val_ratio
SEED = cfg.seed
TILE_SIZE = cfg.minari_tile_size

all_keys = []
for dataset_id in MINARI_DATASETS:
    ds = load_minari_dataset(dataset_id, download=MINARI_DOWNLOAD)
    for ep_id in iterate_episode_ids(ds, MAX_EPISODES_PER_DATASET):
        all_keys.append((dataset_id, ep_id))

train_keys, val_keys = split_minari_episodes(all_keys, VAL_RATIO, SEED)
print(f"Episodes: total={len(all_keys)} train={len(train_keys)} val={len(val_keys)}")

print("Replaying Minari episodes (RGB partial obs)...")
t0 = time.time()
full_cache = precompute_minari_for_keys(
    all_keys,
    download=False,
    tile_size=TILE_SIZE,
    log_every=50,
)
train_cache = filter_minari_cache(full_cache, train_keys)
val_cache = filter_minari_cache(full_cache, val_keys)
train_steps = sum(len(v) for v in train_cache.values())
val_steps = sum(len(v) for v in val_cache.values())
print(f"SFT samples: train={train_steps}, val={val_steps} ({time.time()-t0:.1f}s)")

train_ds = MiniGridSFTDataset.from_minari(
    MINARI_DATASETS,
    tokenizer,
    image_processor,
    trajectory_cache=train_cache,
    download=False,
    tile_size=TILE_SIZE,
)
val_ds = MiniGridSFTDataset.from_minari(
    MINARI_DATASETS,
    tokenizer,
    image_processor,
    trajectory_cache=val_cache,
    download=False,
    tile_size=TILE_SIZE,
)
collator = MiniGridSFTCollator(tokenizer, model.cfg.lm_max_length)
_probe = collator([train_ds[0]])
print("Collator keys:", sorted(_probe.keys()))

sample = train_ds[0]
print("\nПример промпта:")
print(sample["text_data"])
print("Ответ:", sample["answer"].strip())

In [ ]:
plt.imshow(sample["image"].permute(1, 2, 0).numpy())
plt.title(f"object_id={sample['object_id']} step={sample['step_index']} action={sample['action']}")
plt.axis("off")
plt.show()

## 5. Стратегия fine-tuning (только верхние слои)

По умолчанию в nanoVLM можно обучать **все** параметры, но для стабильного и быстрого fine-tuning мы замораживаем почти всё.

| Что | Как |
|-----|-----|
| Инициализация | `VisionLanguageModel.from_pretrained(...)` — **полные** веса VLM (vision + LM + MP) |
| Обучаем | **Только верхние слои**: последние блоки ViT + последние блоки LM + `MP` + `lm_head`/нормы |
| Замораживаем | Остальные слои vision+LM (и эмбеддинги) |
| LR группы | MP (`lr_mp`) выше, “верхние backbone-слои” (`lr_backbones`) ниже |
| Schedule | Cosine decay + 3% warmup |
| Loss | Embedding log-loss + CE по токенам ответа; eval: emb-acc и gen-acc (greedy, 3 действия) |
| AMP | `bfloat16` на CUDA, иначе float32 |

In [ ]:
# --- гиперпараметры ---
EPOCHS = 3
BATCH_SIZE = 8 if torch.cuda.is_available() else 2
GRAD_ACCUM = 1

# Fine-tune только верхние слои
UNFREEZE_VIT_BLOCKS = 2   # сколько последних ViT блоков обучать
UNFREEZE_LM_BLOCKS = 2    # сколько последних LM блоков обучать

LR_MP = 1e-3
LR_BACKBONES = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
LOG_EVERY = 10
EVAL_EVERY = 50
CE_LOSS_WEIGHT = 1.0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
amp_dtype = torch.bfloat16


def set_requires_grad(module, value: bool) -> None:
    for p in module.parameters():
        p.requires_grad = value


def freeze_for_sft(model: VisionLanguageModel, *, vit_last_n: int, lm_last_n: int) -> None:
    # 1) Freeze everything
    set_requires_grad(model, False)

    # 2) Always train modality projector
    set_requires_grad(model.MP, True)

    # 3) Unfreeze last N ViT blocks + final layer norm
    vit = model.vision_encoder
    if hasattr(vit, "blocks"):
        n = len(vit.blocks)
        for i in range(max(0, n - vit_last_n), n):
            set_requires_grad(vit.blocks[i], True)
    if hasattr(vit, "layer_norm"):
        set_requires_grad(vit.layer_norm, True)

    # 4) Unfreeze last N LM blocks + final norm + head
    dec = model.decoder
    if hasattr(dec, "blocks"):
        n = len(dec.blocks)
        for i in range(max(0, n - lm_last_n), n):
            set_requires_grad(dec.blocks[i], True)
    if hasattr(dec, "norm"):
        set_requires_grad(dec.norm, True)
    if hasattr(dec, "head"):
        set_requires_grad(dec.head, True)


freeze_for_sft(model, vit_last_n=UNFREEZE_VIT_BLOCKS, lm_last_n=UNFREEZE_LM_BLOCKS)

n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params: {n_trainable:,} / {sum(p.numel() for p in model.parameters()):,}")

import importlib
import minigrid_collator as _mc
importlib.reload(_mc)
from minigrid_collator import MiniGridSFTCollator, batch_prompt_tensors

collator = MiniGridSFTCollator(tokenizer, model.cfg.lm_max_length)
_probe = collator([train_ds[0]])
print("Collator keys:", sorted(_probe.keys()))

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
    drop_last=len(train_ds) >= BATCH_SIZE,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
)

# Optimizer: берём только параметры с requires_grad=True
mp_params = [p for p in model.MP.parameters() if p.requires_grad]
backbone_params = [p for p in list(model.decoder.parameters()) + list(model.vision_encoder.parameters()) if p.requires_grad]

param_groups = [
    {"params": mp_params, "lr": LR_MP},
    {"params": backbone_params, "lr": LR_BACKBONES},
]
optimizer = optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)
model.to(device)

max_steps = max(1, (len(train_loader) // GRAD_ACCUM) * EPOCHS)
print(f"Device: {device} | batches/epoch: {len(train_loader)} | max_steps: {max_steps}")

## 6. Обучение SFT + метрики

In [ ]:
import importlib
import minigrid_collator as _mc
importlib.reload(_mc)
from minigrid_collator import MiniGridSFTCollator, batch_prompt_tensors, last_hidden_after_prompt

from action_logloss import (
    action_embedding_matrix,
    action_first_token_ids,
    count_allowed_hits,
    optimal_set_log_loss,
    positive_action_mask,
    predict_actions_from_hidden,
    actions_from_first_token_ids,
)


def get_lr(step: int, max_lr: float, max_steps: int) -> float:
    min_lr = max_lr * 0.1
    warmup = max(1, int(max_steps * 0.03))
    if step < warmup:
        return max_lr * (step + 1) / warmup
    if step >= max_steps:
        return min_lr
    decay = (step - warmup) / max(max_steps - warmup, 1)
    return min_lr + 0.5 * (1 + math.cos(math.pi * decay)) * (max_lr - min_lr)


def first_grid_action(text: str) -> str:
    """First token if it is exactly left/right/forward (no substring heuristic)."""
    t = (text or "").strip().lower().split()
    if not t:
        return ""
    w = t[0].rstrip(".,;:!?")
    return w if w in ("left", "right", "forward") else ""


@torch.no_grad()
def eval_metrics(model, tokenizer, val_loader, device, max_new_tokens=8):
    model.eval()
    total_loss, n_batches = 0.0, 0
    emb_hits, gen_hits, n_samples = 0, 0, 0
    action_emb = action_embedding_matrix(model, tokenizer, device)
    first_action_toks = action_first_token_ids(tokenizer, device)

    for batch in val_loader:
        images = batch["image"].to(device)
        prompt_ids, prompt_mask = batch_prompt_tensors(
            batch,
            tokenizer=tokenizer,
            max_length=model.cfg.lm_max_length,
            device=device,
        )
        allowed = batch.get("allowed_actions")
        h = last_hidden_after_prompt(model, prompt_ids, images, attention_mask=prompt_mask)
        pos = positive_action_mask(allowed, h.size(0), device, dtype=h.dtype)
        total_loss += optimal_set_log_loss(h, action_emb, pos).item()
        n_batches += 1
        emb_hits += count_allowed_hits(predict_actions_from_hidden(h, action_emb), allowed)
        gen = model.generate(
            prompt_ids,
            images,
            prompt_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            restrict_first_token_to=first_action_toks,
        )
        gen_preds = actions_from_first_token_ids(gen[:, 0], first_action_toks)
        gen_hits += count_allowed_hits(gen_preds, allowed)
        n_samples += h.size(0)

    model.train()
    loss = total_loss / max(n_batches, 1)
    emb_acc = emb_hits / max(n_samples, 1)
    gen_acc = gen_hits / max(n_samples, 1)
    return loss, emb_acc, gen_acc


history = {"step": [], "train_loss": [], "val_loss": [], "val_emb_acc": [], "val_gen_acc": []}
global_step = 0
best_val_acc = -1.0
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Пересоздаём loaders с актуальным collator (на случай, если ячейка 5 не перезапускалась)
collator = MiniGridSFTCollator(tokenizer, model.cfg.lm_max_length)
train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
    drop_last=len(train_ds) >= BATCH_SIZE,
)
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0,
    pin_memory=device.type == "cuda",
)
_probe = collator([train_ds[0]])
assert "prompt_input_ids" in _probe or "prompt_texts" in _probe, _probe.keys()
print("Collator keys:", sorted(_probe.keys()))

print("=" * 60)
print("SFT: embedding log-loss + CE on answer tokens")
print("=" * 60)

for epoch in range(EPOCHS):
    model.train()
    epoch_loss, epoch_batches = 0.0, 0
    t_epoch = time.time()

    for batch_idx, batch in enumerate(train_loader):
        images = batch["image"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        prompt_ids, prompt_mask = batch_prompt_tensors(
            batch,
            tokenizer=tokenizer,
            max_length=model.cfg.lm_max_length,
            device=device,
        )
        allowed = batch.get("allowed_actions")

        optimizer.zero_grad(set_to_none=True)
        action_emb = action_embedding_matrix(model, tokenizer, device)

        if use_amp:
            with torch.autocast(device_type="cuda", dtype=amp_dtype):
                h = last_hidden_after_prompt(model, prompt_ids, images, attention_mask=prompt_mask)
                pos = positive_action_mask(allowed, h.size(0), device, dtype=h.dtype)
                emb_loss = optimal_set_log_loss(h, action_emb, pos)
                _, ce_loss = model(input_ids, images, attention_mask, targets=labels)
                loss = emb_loss + CE_LOSS_WEIGHT * ce_loss
        else:
            h = last_hidden_after_prompt(model, prompt_ids, images, attention_mask=prompt_mask)
            pos = positive_action_mask(allowed, h.size(0), device, dtype=h.dtype)
            emb_loss = optimal_set_log_loss(h, action_emb, pos)
            _, ce_loss = model(input_ids, images, attention_mask, targets=labels)
            loss = emb_loss + CE_LOSS_WEIGHT * ce_loss

        (loss / GRAD_ACCUM).backward()

        if (batch_idx + 1) % GRAD_ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.param_groups[0]["lr"] = get_lr(global_step, LR_MP, max_steps)
            optimizer.param_groups[1]["lr"] = get_lr(global_step, LR_BACKBONES, max_steps)
            optimizer.step()
            global_step += 1

        loss_val = loss.item()
        epoch_loss += loss_val
        epoch_batches += 1

        if global_step > 0 and global_step % LOG_EVERY == 0:
            print(
                f"[ep {epoch+1}/{EPOCHS}] step {global_step} | "
                f"logloss={loss_val:.4f} | lr_mp={optimizer.param_groups[0]['lr']:.2e} "
                f"lr_bb={optimizer.param_groups[1]['lr']:.2e}"
            )
            history["step"].append(global_step)
            history["train_loss"].append(loss_val)

        if global_step > 0 and global_step % EVAL_EVERY == 0:
            val_loss, val_emb_acc, val_gen_acc = eval_metrics(model, tokenizer, val_loader, device)
            print(
                f"  >> val_logloss={val_loss:.4f} | val_emb_acc={val_emb_acc*100:.2f}% "
                f"| val_gen_acc={val_gen_acc*100:.2f}%"
            )
            history["val_loss"].append(val_loss)
            history["val_emb_acc"].append(val_emb_acc)
            history["val_gen_acc"].append(val_gen_acc)
            if val_gen_acc > best_val_acc:
                best_val_acc = val_gen_acc
                model.save_pretrained(str(SFT_OUTPUT_DIR / "best"))
                print(f"  >> saved best -> {SFT_OUTPUT_DIR / 'best'}")

    val_loss, val_emb_acc, val_gen_acc = eval_metrics(model, tokenizer, val_loader, device)
    print(
        f"Epoch {epoch+1} ({time.time()-t_epoch:.0f}s) | "
        f"avg_train_logloss={epoch_loss/max(epoch_batches,1):.4f} | "
        f"val_logloss={val_loss:.4f} | val_emb_acc={val_emb_acc*100:.2f}% "
        f"| val_gen_acc={val_gen_acc*100:.2f}%"
    )
    model.save_pretrained(str(SFT_OUTPUT_DIR / f"epoch_{epoch+1}"))

model.save_pretrained(str(SFT_OUTPUT_DIR / "last"))
print("=" * 60)
print(f"Done. Best val acc: {best_val_acc*100:.2f}%")
print(f"Checkpoints: {SFT_OUTPUT_DIR}")
print("=" * 60)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if history["step"]:
    axes[0].plot(history["step"], history["train_loss"], label="train loss")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("loss")
    axes[0].set_title("Training loss")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

eval_steps = history["step"][-len(history["val_gen_acc"]):] if history["val_gen_acc"] else []
if history["val_gen_acc"]:
    axes[1].plot(eval_steps, [a * 100 for a in history["val_gen_acc"]], "o-", color="green", label="val gen acc %")
    if history["val_emb_acc"]:
        axes[1].plot(eval_steps, [a * 100 for a in history["val_emb_acc"]], "s-", color="blue", alpha=0.7, label="val emb acc %")
    if history["val_loss"]:
        ax2 = axes[1].twinx()
        ax2.plot(eval_steps, history["val_loss"], "s--", color="red", alpha=0.7, label="val loss")
        ax2.set_ylabel("val loss", color="red")
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("accuracy %")
    axes[1].set_title("Validation metrics")
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Inference: проверка обученной модели

In [ ]:
ckpt = SFT_OUTPUT_DIR / "best"
if not ckpt.exists():
    ckpt = SFT_OUTPUT_DIR / "last"

print(f"Loading fine-tuned weights: {ckpt}")
ft_model = VisionLanguageModel.from_pretrained(str(ckpt)).to(device)
ft_model.eval()

test_item = val_ds[0]
image = test_item["image"].unsqueeze(0).to(device)
prompt = test_item["text_data"]
gold = test_item["action"]

enc = tokenizer([prompt], padding=True, return_tensors="pt", truncation=True,
                max_length=ft_model.cfg.lm_max_length)
input_ids = enc["input_ids"].to(device)
attention_mask = enc["attention_mask"].to(device)

from action_logloss import action_first_token_ids

first_action_toks = action_first_token_ids(tokenizer, device)
with torch.no_grad():
    gen = ft_model.generate(
        input_ids,
        image,
        attention_mask,
        max_new_tokens=8,
        do_sample=False,
        restrict_first_token_to=first_action_toks,
    )
pred = tokenizer.batch_decode(gen, skip_special_tokens=True)[0]


def first_grid_action(text: str) -> str:
    t = (text or "").strip().lower().split()
    if not t:
        return ""
    w = t[0].rstrip(".,;:!?")
    return w if w in ("left", "right", "forward") else ""


allowed = set(test_item.get("allowed_actions") or [gold])
pa = first_grid_action(pred)
print(f"Gold: {gold}")
print(f"Pred: {pred}")
print(f"In allowed optimal set: {pa in allowed} (parsed: {pa!r})")

plt.imshow(test_item["image"].permute(1, 2, 0).numpy())
plt.title(f"pred={pa!r} | gold={gold}")
plt.axis("off")
plt.show()

In [ ]:
# Визуализация sample: поле + позиция агента + score в соседних клетках
import numpy as np

from maze_expert import make_env_from_object, rollout_expert_trajectory


def _cell_distances(layout: dict) -> dict[tuple[int, int], int]:
    """BFS distance on cells (ignore direction)."""
    size = layout["size"]
    walls = {tuple(w) for w in layout["walls"]}
    goal = tuple(layout["goal_pos"])

    def walkable(c):
        x, y = c
        return 0 <= x < size and 0 <= y < size and (x, y) not in walls

    from collections import deque

    q = deque([goal])
    dist = {goal: 0}
    while q:
        x, y = q.popleft()
        for dx, dy in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            nx, ny = x + dx, y + dy
            if not walkable((nx, ny)):
                continue
            if (nx, ny) in dist:
                continue
            dist[(nx, ny)] = dist[(x, y)] + 1
            q.append((nx, ny))
    return dist


def show_sample_with_move_scores(obj: dict, *, step_index: int = 0):
    """
    Рисует поле лабиринта и позицию агента на выбранном step_index.

    В соседних клетках отображает score за перемещение в эту клетку:
      score = dist(cur) - dist(neighbor)
    (положительный = ближе к цели, отрицательный = дальше).
    """
    layout = obj["layout"]
    size = layout["size"]
    walls = {tuple(w) for w in layout["walls"]}
    goal = tuple(layout["goal_pos"])

    # Восстановим состояние агента на step_index через экспертную траекторию
    traj = rollout_expert_trajectory(obj)
    step_index = int(np.clip(step_index, 0, len(traj) - 1))

    # Для точной позиции используем env, чтобы корректно учесть повороты
    env = make_env_from_object(obj)
    obs, _ = env.reset(seed=obj["seed"])
    for i in range(step_index):
        env.step({"left": 0, "right": 1, "forward": 2}[traj[i].action])

    ax, ay = int(env.unwrapped.agent_pos[0]), int(env.unwrapped.agent_pos[1])
    env.close()

    dist = _cell_distances(layout)
    cur_d = dist.get((ax, ay), None)

    grid = np.zeros((size, size), dtype=np.int32)
    for (x, y) in walls:
        grid[y, x] = 1

    import matplotlib.pyplot as plt

    fig, axp = plt.subplots(1, 1, figsize=(6, 6))
    axp.imshow(grid, cmap="gray_r", origin="upper")

    # Walls and grid lines
    axp.set_xticks(np.arange(-0.5, size, 1), minor=True)
    axp.set_yticks(np.arange(-0.5, size, 1), minor=True)
    axp.grid(which="minor", color="black", linewidth=0.5, alpha=0.25)
    axp.set_xticks([])
    axp.set_yticks([])

    # Goal + agent
    axp.text(goal[0], goal[1], "G", ha="center", va="center", fontsize=14, fontweight="bold", color="green")
    axp.text(ax, ay, "A", ha="center", va="center", fontsize=14, fontweight="bold", color="blue")

    # Neighbor scores (4-neighborhood)
    for dx, dy, sym in ((1, 0, "→"), (-1, 0, "←"), (0, 1, "↓"), (0, -1, "↑")):
        nx, ny = ax + dx, ay + dy
        if (nx, ny) in walls or not (0 <= nx < size and 0 <= ny < size):
            continue
        nd = dist.get((nx, ny), None)
        if cur_d is None or nd is None:
            score = None
        else:
            score = cur_d - nd
        txt = f"{sym}\n{score:+d}" if score is not None else f"{sym}\nNA"
        axp.text(nx, ny, txt, ha="center", va="center", fontsize=9, color="darkred")

    axp.set_title(f"obj={obj.get('id')} step={step_index} | pos=({ax},{ay}) | dist={cur_d}")
    plt.show()


# Пример вызова
show_sample_with_move_scores(dataset["objects"][0], step_index=0)

## Следующий шаг

Чекпоинт `checkpoints/sft-minigrid/best` используйте как инициализацию для **GRPO**-дообучения в среде MiniGrid.